# 04. Аудио: waveform → mel → VAE → vocoder

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 4](https://diffusion.csail.mit.edu/2026/docs/20260128_Lecture_04_edited.pdf).
<!-- study-navigation:end -->

**Результат:** понять частотное представление, роль audio VAE и vocoder, проверить duration/scaling/clipping

**Перед началом:** K6 03; базовая вероятность Gaussian. [Порядок всех материалов](../../curriculum/CONTENTS.md).
[Курс K6](../README.md) · [Покомпонентная карта](../COMPONENT_GUIDE.md) · [Предыдущий опыт K5](../../kandinsky-5/notebooks/11_kvae_separate.ipynb).

Основной архитектурный источник: [Kandinsky 6 на Hugging Face Papers](https://huggingface.co/papers/2610.05608),
[полный отчёт, §3.1](https://arxiv.org/html/2610.05608v1). Открывайте статью рядом с notebook.
Мы различаем опубликованную архитектуру, конкретный checkpoint config и результат нашего опыта.

**Код для сравнения (закреплённая revision `7ceeefad`):**
- [`core/components/vae_audio.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/components/vae_audio.py)
- [`core/components/mmaudio/features_utils.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/components/mmaudio/features_utils.py)
- [`core/algo/postprocess_audio.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/algo/postprocess_audio.py)

Run All выполняет небольшие CPU-опыты. Случайная уменьшенная сеть проверяет вычисления;
pretrained генерацию, качество голоса и производительность M3 она не измеряет.
Большие веса не скачиваются, ComfyUI jobs не отправляются. В Colab первая ячейка получает
исходники курса из GitHub, если их ещё нет; пакеты не устанавливает. На Mac используется существующий checkout.
Ниже объяснение идёт перед каждым опытом.


In [ ]:
from pathlib import Path
import json, sys, os, math
for parent in (Path.cwd(), *Path.cwd().parents):
    candidates = (parent, parent / 'kandinsky-6')
    found = next((p for p in candidates if (p / 'labkit/components.py').is_file()), None)
    if found is not None:
        ROOT = found.resolve()
        break
else:
    if not Path('/content').is_dir():
        raise FileNotFoundError('Откройте notebook внутри checkout generative-ai-research.')
    import subprocess
    checkout=Path('/content/generative-ai-research-k6-components')
    if not checkout.exists():
        subprocess.run(['git','clone','--depth','1','--branch',
            os.environ.get('K6_LABS_REF','main'),
            'https://github.com/vladislav-vasilenko/generative-ai-research.git',str(checkout)],check=True)
    ROOT=checkout/'kandinsky-6'
    if not (ROOT/'labkit/components.py').is_file():
        raise FileNotFoundError('В выбранной revision нет component labs; проверьте K6_LABS_REF.')
sys.path.insert(0, str(ROOT))
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / 'artifacts/component_labs/mpl-cache'))
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from labkit.components import (config, native, show_source, native_causal_conv,
    tiny_model, tiny_inputs, activate_gates, fresh_forward, write_metrics, figure_path)
torch.set_num_threads(2)
torch.manual_seed(42)
np.random.seed(42)
LAB = '04_audio_vae_and_vocoder'
metrics = {}
print('CPU teaching mode; PyTorch', torch.__version__)


## 1. Три пространства и три разные задачи

Первичные статьи компонентов: [MMAudio](https://arxiv.org/abs/2412.15322),
[BigVGAN](https://arxiv.org/abs/2206.04658). Их полный generation backbone не загружается вместо K6.

Waveform — амплитуда во времени. STFT — комплексная частотно-временная матрица; mel-проекция
сжимает частотные полосы. VAE преобразует mel в 40-channel latent. Decoder VAE восстанавливает mel,
но не waveform: эту задачу решает BigVGAN-style vocoder.

Код K6: `FeaturesUtils.wrapped_encode` возвращает mean posterior; `wrapped_decode` — mel.
Режим checkpoint — 44100 Hz, downsample factor 1024, scale Lite Distill=0.417.
Обычный Lite config использует другой scale (0.5302). Не копируйте нормализацию между checkpoints.
KVAE-Audio — отдельный codec, не автоматическая замена MMAudio latent contract.

### Почему STFT inverse ещё не vocoder

Если сохранить phase, inverse STFT может точно вернуть синтетический waveform.
Mel magnitude сам по себе фазу не сохраняет; обученный vocoder восполняет недостающую информацию.
Наш опыт ниже не воспроизводит MMAudio preprocessing целиком.


In [ ]:
sr=44100;time=torch.arange(sr//4)/sr
wave=.2*torch.sin(2*torch.pi*440*time)+.1*torch.sin(2*torch.pi*880*time)
window=torch.hann_window(1024)
stft=torch.stft(wave,n_fft=1024,hop_length=256,window=window,return_complex=True)
inverse=torch.istft(stft,n_fft=1024,hop_length=256,window=window,length=wave.numel())
error=(wave-inverse).abs().max().item();assert error<1e-5
# Educational triangular mel filters; not the exact checkpoint feature extractor.
freqs=torch.linspace(0,sr/2,513);mel=lambda f:2595*torch.log10(1+f/700)
points=700*(10**(torch.linspace(0,mel(torch.tensor(sr/2)).item(),130)/2595)-1)
left=(freqs[None,:]-points[:-2,None])/(points[1:-1,None]-points[:-2,None])
right=(points[2:,None]-freqs[None,:])/(points[2:,None]-points[1:-1,None])
bank=torch.minimum(left,right).clamp(min=0)
mel_power=bank@stft.abs().square()
metrics['stft_inverse_max_error']=error
fig,axes=plt.subplots(1,2,figsize=(10,3));axes[0].plot(time[:500],wave[:500])
axes[0].set_title('Synthetic waveform');axes[1].imshow(torch.log1p(mel_power),origin='lower',aspect='auto')
axes[1].set_title('Educational mel energy, 128 bands');fig.tight_layout()
fig.savefig(figure_path(LAB,'audio_spaces'));plt.show()


## 2. Сколько audio tokens соответствует видео

Длительность код вычисляет через число RGB frames/fps, затем берёт ceil(samples/1024).
Это даёт запас меньше одного audio latent step; он нужен для decoder/mux boundaries.
Интервал между первым и последним video timestamp и длительность хранения N frames/fps отличаются.


In [ ]:
prepare=native('core.algo.prepare_latents')
rows=[]
for latent_frames in [1,9,31]:
    frames=(latent_frames-1)*4+1
    n=prepare.audio_latent_duration(latent_frames)
    delta=n*1024/44100-frames/24
    assert 0<=delta<1024/44100
    rows.append({'video_frames':frames,'audio_latents':n,'padding_seconds':delta})
print(rows);metrics['duration_contract']=rows


## 3. Настоящий postprocess с явно учебными decoder/vocoder

Перед decoder выполняется обратная нормализация. Чтобы проверить эту обвязку без весов,
подставим test doubles: mel decoder возвращает вход, vocoder — заданный waveform.
Так исполняется настоящий `postprocess_audio`, но нет заявления о качестве звука.
`clip` сохраняет масштаб до насыщения; `normalize` меняет громкость всего примера по peak.


In [ ]:
post=native('core.algo.postprocess_audio');Bundle=native('core.types').LatentBundle
class DecoderDouble:
    scaling_factor=.417
    def wrapped_decode(self,z):
        self.seen=z.clone();return z
class VocoderDouble:
    def __call__(self,mel):return torch.tensor([[[-2.,-.2,0.,.2,2.]]])
decoder=DecoderDouble();bundle=Bundle(None,torch.ones(1,5,40)*.417,None,torch.tensor([0,5],dtype=torch.int32))
clipped=post.postprocess_audio(bundle,decoder,VocoderDouble(),'clip')[0]
normalized=post.postprocess_audio(bundle,decoder,VocoderDouble(),'normalize')[0]
assert torch.allclose(decoder.seen,torch.ones_like(decoder.seen))
assert clipped[1]!=normalized[1]
metrics['clip_pcm']=clipped.tolist();metrics['normalize_pcm']=normalized.tolist()
print('clip:',clipped,'normalize:',normalized)


## Самостоятельное исследование

1. Удалите phase и объясните, почему mel→waveform не единственное обратное преобразование.
2. Измените scale с .417 на .5302 в одном месте и измерьте ошибку latent roundtrip.
3. Сравните peak и RMS до/после normalize. Почему peak normalization осложняет A/B громкости?

Перед изменением запишите гипотезу. Сохраните baseline, меняйте один фактор и повторяйте по 3–5 seeds. Файл metrics.json содержит измерение конкретного запуска, а не статистическую оценку качества модели.


In [ ]:
write_metrics(LAB, metrics)


## Критерий понимания

Вы можете назвать, что возвращает каждый audio component. 400 выходов DiT — не 400 channels audio VAE:
это 10 grid points × 40 channels, которые разберём в π-Flow лабораторной.

[Вернуться к маршруту K6](../README.md) · [Полное содержание](../../curriculum/CONTENTS.md) · [Шаблон отчёта](../../curriculum/EXPERIMENT_TEMPLATE.md).


[← 03. Видео VAE: posterior, compression, causality и scaling](03_video_vae.ipynb) · [05. Строительные блоки: patches, time, AdaLN и gates →](05_nn_patches_and_modulation.ipynb)
